# Day 24 — Data Transformation
Project 1: Retail Sales & Customer Performance Analysis

Objective: Create new columns that make analysis easier. Source: Day 24 Task For DA&DS(2).pdf; input: Day 23 cleaned dataset.

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display
df = pd.read_csv("cleaned_retail_sales.csv", parse_dates=["Order_Date"])
original = df.copy()
print("Before:", df.shape)
display(df.head())

Before: (30, 11)


,Order_ID,Order_Date,Customer_Name,City,Category,Product,Quantity,Unit_Price,Sales,Customer_Rating,Customer_Rating_Was_Imputed
0,ORD001,2026-10-02,Rahul,Mumbai,Clothing,T-Shirt,2,650,1300,5.0,False
1,ORD002,2026-10-03,Neha,Nagpur,Grocery,Rice,3,320,960,3.0,False
2,ORD003,2026-10-04,Vikram,Nashik,Home,Lamp,4,1100,4400,4.0,False
3,ORD004,2026-10-05,Priya,Pune,Electronics,Headphones,5,1500,7500,5.0,False
4,ORD005,2026-10-06,Arjun,Pune,Clothing,T-Shirt,1,650,650,4.0,False


## Task 1 — Date Features

In [2]:
df["Year"] = df["Order_Date"].dt.year
df["Month"] = df["Order_Date"].dt.month
df["Month_Name"] = df["Order_Date"].dt.month_name()
df["Quarter"] = df["Order_Date"].dt.quarter
display(df[["Order_Date","Year","Month","Month_Name","Quarter"]].head())

,Order_Date,Year,Month,Month_Name,Quarter
0,2026-10-02,2026,10,October,4
1,2026-10-03,2026,10,October,4
2,2026-10-04,2026,10,October,4
3,2026-10-05,2026,10,October,4
4,2026-10-06,2026,10,October,4


## Task 2 — Profit Margin
Profit Margin = Profit / Sales × 100. The source lacks Profit and Cost. Use missing values, not zero or invented profit. Zero sales also makes the ratio undefined.

In [3]:
if "Profit" in df.columns:
    df["Profit_Margin"] = df["Profit"].div(df["Sales"].replace(0, np.nan))*100
else:
    df["Profit_Margin"] = np.nan
    print("Profit margin unavailable: Profit column absent.")
print("Available margins:", df["Profit_Margin"].count())

Profit margin unavailable: Profit column absent.
Available margins: 0


## Task 3 — Order Value
Quantity × Unit Price is the line value; each order ID is unique in this dataset.

In [4]:
df["Order_Value"] = df["Quantity"] * df["Unit_Price"]
print("Unique order IDs:", df["Order_ID"].is_unique)
display(df[["Order_ID","Quantity","Unit_Price","Sales","Order_Value"]].head())

Unique order IDs: True


,Order_ID,Quantity,Unit_Price,Sales,Order_Value
0,ORD001,2,650,1300,1300
1,ORD002,3,320,960,960
2,ORD003,4,1100,4400,4400
3,ORD004,5,1500,7500,7500
4,ORD005,1,650,650,650


## Task 4 — Customer Segmentation
Use total observed spending per Customer_Name (a proxy, not a verified customer ID). Illustrative fixed business thresholds: High Value ≥ INR 15,000; Medium Value ≥ INR 10,000 and below INR 15,000; Low Value below INR 10,000. These are analyst-defined, not institute-mandated or lifetime-value thresholds.

In [5]:
df["Customer_Total_Spending"] = df.groupby("Customer_Name")["Sales"].transform("sum")
df["Customer_Segment"] = np.select([df.Customer_Total_Spending >= 15000, df.Customer_Total_Spending >= 10000], ["High Value", "Medium Value"], default="Low Value")
customers = df[["Customer_Name","Customer_Total_Spending","Customer_Segment"]].drop_duplicates().sort_values("Customer_Total_Spending", ascending=False)
display(customers)
display(customers.Customer_Segment.value_counts())

,Customer_Name,Customer_Total_Spending,Customer_Segment
3,Priya,17780,High Value
5,Asha,14240,Medium Value
2,Vikram,13350,Medium Value
0,Rahul,13350,Medium Value
4,Arjun,12450,Medium Value
1,Neha,10700,Medium Value


Customer_Segment
Medium Value    5
High Value      1
Name: count, dtype: int64

## Task 5 — Additional Analytical Columns
Sales_Category: High ≥ 3000, Medium ≥ 1000, otherwise Low (INR per line). Customer_Type: Repeat if more than one distinct order in the observed period. Product_Performance: High if product revenue ≥ median product revenue, otherwise Low. Order_Size: Large ≥ 5 units, Medium ≥ 3, otherwise Small. Day_Name and Is_Weekend support calendar analysis. Customer_Total_Spending and Product_Total_Sales are repeated group totals; do not sum them across rows. Profit_Category is not calculated because profit is unavailable.

In [6]:
df["Sales_Category"] = np.select([df.Sales >= 3000, df.Sales >= 1000], ["High", "Medium"], default="Low")
df["Customer_Order_Count"] = df.groupby("Customer_Name")["Order_ID"].transform("nunique")
df["Customer_Type"] = np.where(df.Customer_Order_Count > 1, "Repeat", "Single-order")
product_totals = df.groupby("Product").Sales.sum()
df["Product_Total_Sales"] = df.Product.map(product_totals)
df["Product_Performance"] = np.where(df.Product_Total_Sales >= product_totals.median(), "High", "Low")
df["Order_Size"] = np.select([df.Quantity >= 5, df.Quantity >= 3], ["Large", "Medium"], default="Small")
df["Day_Name"] = df.Order_Date.dt.day_name()
df["Is_Weekend"] = df.Order_Date.dt.dayofweek >= 5
print("Product revenue median:", product_totals.median())
display(df[["Sales_Category","Customer_Type","Product_Performance","Order_Size","Day_Name","Is_Weekend"]].head())

Product revenue median: 19575.0


C:\Users\Sarkar\AppData\Local\Temp\ipykernel_30864\1200293053.py:9: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["Is_Weekend"] = df.Order_Date.dt.dayofweek >= 5


,Sales_Category,Customer_Type,Product_Performance,Order_Size,Day_Name,Is_Weekend
0,Medium,Repeat,Low,Small,Friday,False
1,Low,Repeat,Low,Medium,Saturday,True
2,High,Repeat,High,Medium,Sunday,True
3,High,Repeat,High,Large,Monday,False
4,Low,Repeat,Low,Small,Tuesday,False


## Validation and export

In [7]:
assert len(df) == len(original)
pd.testing.assert_frame_equal(df[original.columns], original)
assert np.allclose(df.Order_Value, df.Sales)
assert np.isclose(df.Sales.sum(), original.Sales.sum())
print("Before:", original.shape, "After:", df.shape)
print("Revenue before:", original.Sales.sum(), "Revenue after:", df.Sales.sum())
print("New columns:", len(df.columns)-len(original.columns))
print("Missing values by column:")
display(df.isna().sum())
print("Validation passed")
df.to_csv("transformed_retail_sales.csv", index=False)
customers.to_csv("customer_segments.csv", index=False)

Before: (30, 11) After: (30, 27)
Revenue before: 81870 Revenue after: 81870
New columns: 16
Missing values by column:


Order_ID                        0
Order_Date                      0
Customer_Name                   0
City                            0
Category                        0
Product                         0
Quantity                        0
Unit_Price                      0
Sales                           0
Customer_Rating                 0
Customer_Rating_Was_Imputed     0
Year                            0
Month                           0
Month_Name                      0
Quarter                         0
Profit_Margin                  30
Order_Value                     0
Customer_Total_Spending         0
Customer_Segment                0
Sales_Category                  0
Customer_Order_Count            0
Customer_Type                   0
Product_Total_Sales             0
Product_Performance             0
Order_Size                      0
Day_Name                        0
Is_Weekend                      0
dtype: int64

Validation passed
